# HF2LI frequency sweep (reference script, V5.3)

Reference copy of `SweepZurich_V5.3.ipynb` from the acoustic isolation (geophone) experiment, using
`optomechanics.Zurich.HF2LI`. See *User Guide → HF2LI Frequency Sweeps* in the docs.

Before running, change in the cells below:

- **Sweep plan** (next cell): the `parts` list and `Use_Sets`. Pick an output `range` close to the
  amplitudes; amplitude / range below roughly 1e-5 rounds to 0 on the device and the read-back check fails.
- **Sweep variables**: `Filepath`, `Filename`, `Metadata`, `Notes`.
- **Index logging**: `Index_File` and `Append_To_Index` (set `Append_To_Index = False` if you
  don't keep a master index CSV), and the `relpath` base directory in the index cell.
- **Last cell** saves this notebook and copies it next to the data file; `_nb_name` must match the
  notebook's filename.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from optomechanics.Zurich.HF2LI import HF2LI
import time

In [ ]:
Use_Sets = [1]   # 1-based indices of parts to include; e.g. [2] runs only P2. Start from 1, not 0.
Delay = 0 #hours

# ── Define parts ───────────────────────────────────────────────────────────────
# Add, remove, or reorder dicts freely. Use_Sets picks which ones run.
# Each dict must have:
#   id         - string label shown in the print table and iteration legend
#   averages   - how many times the full enable pattern is repeated
#   amplitudes - list of output amplitudes [V] swept for each enable entry
#   enable     - list of output enable values, one per run through amplitudes
#                (e.g. [0,1,0,1] -> 4 runs: off, on, off, on)
#                or a single value applied to all runs
#   start      - start frequency [Hz]
#   stop       - stop frequency [Hz]
#   range      - output voltage range [V]
#   npts       - number of frequency points
#   BW         - demodulator bandwidth [Hz]
#
# Total iterations per part = averages * len(enable) * len(amplitudes)
globalBW = 1.098#
GlobalPtsDivision = 10
GlobalAvg = 2
LowPwr = 12e-3
HighPwr = [500e-3]
enable = [0,1]
parts = [
    {
        "id":         "P1",
        "averages":   GlobalAvg,
        "amplitudes": [0.05e-3, 0.1e-3, 0.25e-3, 0.5e-3],
        "enable":     enable,   # 4 runs: off, on, off, on
        "start":      20,
        "stop":       2000,
        "range":      10e-3,
        "npts":       int((2000-20)/GlobalPtsDivision)+1,    # (2000-0)/4 + 1
        "BW":         globalBW,
    },



    # {
    #     "id":         "P2",
    #     "averages":   GlobalAvg,
    #     "amplitudes": LowPwr,
    #     "enable":     enable,   # 4 runs: off, on, off, on
    #     "start":      720,
    #     "stop":       760,
    #     "range":      1,
    #     "npts":       int((760-720)/GlobalPtsDivision)+1,    # (2000-0)/4 + 1
    #     "BW":         globalBW,
    # },


    # {
    #     "id":         "P3",
    #     "averages":   GlobalAvg,
    #     "amplitudes": HighPwr,
    #     "enable":     enable,   # 4 runs: off, on, off, on
    #     "start":      760,
    #     "stop":       2000,
    #     "range":      10,
    #     "npts":       int((2000-760)/GlobalPtsDivision)+1,    # (2000-0)/4 + 1
    #     "BW":         globalBW,
    # },
    # {
    #     "id":         "P4",
    #     "averages":   GlobalAvg,
    #     "amplitudes": [LowPwr],
    #     "enable":     enable,   # 4 runs: off, on, off, on
    #     "start":      990,
    #     "stop":       1250,
    #     "range":      1,
    #     "npts":       int((1250-990)/GlobalPtsDivision)+1,    # (2000-0)/4 + 1
    #     "BW":         globalBW,
    # },
    #     {
    #     "id":         "P5",
    #     "averages":   GlobalAvg,
    #     "amplitudes": HighPwr,
    #     "enable":     enable,   # 4 runs: off, on, off, on
    #     "start":      1250,
    #     "stop":       1420,
    #     "range":      1,
    #     "npts":       int((1420-1350)/GlobalPtsDivision)+1,    # (2000-0)/4 + 1
    #     "BW":         globalBW,
    # },
    #     {
    #     "id":         "P6",
    #     "averages":   GlobalAvg,
    #     "amplitudes": [LowPwr],
    #     "enable":     enable,   # 4 runs: off, on, off, on
    #     "start":      1420,
    #     "stop":       2000,
    #     "range":      1,
    #     "npts":       int((2000-1420)/GlobalPtsDivision)+1,    # (2000-0)/4 + 1
    #     "BW":         globalBW,
    # },

]


# ── Build combined lists (only from selected parts) ──────────────────────────
_active_parts = [parts[i - 1] for i in Use_Sets]

Output_Amplitude = []
Output_Enable    = []
Start_Freq_Hz    = []
Stop_Freq_Hz     = []
Amplitude_Range  = []
Num_Points       = []
Bandwidth_Hz     = []
_part_ids        = []  # per-iteration part identifier
_avg_nums        = []  # per-iteration average number (1-based, counts across repeats)

for p in _active_parts:
    amplitudes = p["amplitudes"]
    averages   = p["averages"]
    en_spec    = p["enable"]
    en_list    = en_spec if isinstance(en_spec, list) else [en_spec] * averages

    amps     = [amp for _   in range(averages) for _  in en_list  for amp in amplitudes]
    enable   = [en  for _   in range(averages) for en in en_list  for _   in amplitudes]
    avg_nums = [rep + 1
                for rep in range(averages)
                for _   in en_list
                for _   in amplitudes]
    n = len(amps)

    Output_Amplitude += amps
    Output_Enable    += enable
    _avg_nums        += avg_nums
    Start_Freq_Hz    += [p["start"]] * n
    Stop_Freq_Hz     += [p["stop"]]  * n
    Amplitude_Range  += [p["range"]] * n
    Num_Points       += [p["npts"]]  * n
    Bandwidth_Hz     += [p["BW"]]    * n
    _part_ids        += [p["id"]]    * n


print(f"{'i':>4}  {'Part':4}  {'Avg':>4}  {'Amp(V)':>12}  {'Enable':>8}  {'Start':>5}  {'Stop':>5}  {'Rng':>4}  {'Npts':>5}  {'Step':>6}  {'BW(Hz)':>8}")
print("-" * 91)
for i, (amp, en, fs, fe, rng, npts, bw) in enumerate(zip(
        Output_Amplitude, Output_Enable, Start_Freq_Hz, Stop_Freq_Hz, Amplitude_Range, Num_Points, Bandwidth_Hz)):
    part = _part_ids[i]
    avg  = _avg_nums[i]
    step = (fe - fs) / (npts - 1)
    print(f"{i:4d}  {part:4}  {avg:4d}  {amp:8.5f}  {en:2d}  {fs:5d}  {fe:5d}  {rng:.4f}  {npts:5d}  {step:6.3f}  {bw:8.5f}")


time.sleep(Delay*3600)

# HF2LI Sweep Variables

Any variable in the cell below may be a single value (default) **or** a Python `list`. If any are lists they must all share the same length `N`, and the sweep is run `N` times. The save mode is decided by `Filename`:

* scalar `Filename` → single `.h5` file containing iterations as datasets `000/`, `001/`, ...
* list `Filename` → one `.h5` file per item (`Filepath` may also be a list to put each file in its own directory).

In [ ]:

Start_Freq_Hz       = Start_Freq_Hz    # Start frequency [Hz]
Stop_Freq_Hz        = Stop_Freq_Hz        # Stop frequency [Hz]
Num_Points          = Num_Points          # Number of frequency points
Demod_Index         = 0      # Demodulator channel to read (0-5)
Settling_Time_s     = None   # Minimum settling time per point [s]; None = device default
Settling_Inaccuracy = 500e-6   # Settling inaccuracy threshold (0-1); None = device default
Averaging_Samples   = 1      # Minimum Number of samples averaged per point
Bandwidth_Control   = 1      # Bandwidth control mode: 0 = manual (uses demod's existing TC, ignores Bandwidth_Hz), 1 = fixed (applies Bandwidth_Hz), 2 = auto (sweeper picks per point)
Bandwidth_Hz        = Bandwidth_Hz   # Per-iteration demodulator bandwidth [Hz]; P1=1.098 Hz, P2=56.43 mHz
Filter_Order        = 4      # Demodulator low-pass filter order (1-8); None = leave unchanged
Log_Sweep           = False  # True = logarithmic frequency spacing; False = linear
Timeout_s           = np.inf # Maximum time to wait for sweep completion [s]
Progress_Interval_s = 0.5     # Time between live plot updates [s]
Output_Channel      = 0      # Output channel index (0 or 1); None = don't configure output
Amplitude_Range     = Amplitude_Range   # Output voltage range [V] (0.01, 0.1, 1, or 10); sets DAC headroom
Output_Amplitude    = Output_Amplitude   # Output amplitude [V] (absolute); converted to fraction of Amplitude_Range on device
Output_Enable       = Output_Enable      # True = enable output channel, False = disable, None = leave unchanged
Output_Osc_Enable   = 1      # True = route oscillator signal to output, False = disable, None = leave unchanged
Output_Osc_Index    = 0      # Oscillator index (0 or 1); output 1 uses osc 0, output 2 uses osc 1

Save                = True   # Save result to HDF5 file after sweep
Filepath            = r"D:\Acoustic isolation\Data\geophone\Vibration Measurements\TwoMassWithWeightNewSpringsAndExpPlate\LinearityTestsIsoPlate"  # Directory for saved HDF5 file
Filename            = "LinearityTest.h5"                                # HDF5 filename (must end in .h5)
Metadata            = {
    "geophone Location":"ISOPlate",
    "Cicuit":"Geophone->100KHzLPFEF502->AmpOnStall->LongCable->Lockin,Lockin-longcable-speaker; Zurich->PowerAmp(*5)->PiezoISOPlate",
    "AmpFilter":"12dB300HzHPF",
    "FilterFreq":"300Hz",
    "GainMode":"HighBW",
    "Gain":"20nA/V",
    "amplitude_range": Amplitude_Range,
                       }   # Extra key/value pairs written to the HDF5 file root
  

# --- Index logging (appended to CSV after sweep completes) ---
Index_File          = r"D:\Acoustic isolation\Data\geophone\index.csv"  # Master index CSV
Append_To_Index     = True            # Append a row to Index_File after a successful sweep
Amplification_dB    = ""              # e.g. 20, 40; leave "" if unknown / not applicable
Amplification_Type  = "high_bandwidth"              # "low_noise" or "high_bandwidth"; "" if unknown
Drive_Source        = "Speaker 3 Fridge"              # "speaker" or "piezo"; "" if unknown
Notes               = "Speaker3, TwoISOMasses, new springs setup, with extra brass mass on exp plate. Linearity Test. Power amp set for 3x Gain."
# add notes to meta data:
Metadata.update({"notes":Notes})


print(f"{
    {
    "Start_Freq_Hz":       Start_Freq_Hz,
    "Stop_Freq_Hz":        Stop_Freq_Hz,
    "Num_Points":          Num_Points,
    "Demod_Index":         Demod_Index,
    "Settling_Time_s":     Settling_Time_s,
    "Settling_Inaccuracy": Settling_Inaccuracy,
    "Averaging_Samples":   Averaging_Samples,
    "Bandwidth_Control":   Bandwidth_Control,
    "Bandwidth_Hz":        Bandwidth_Hz,
    "Filter_Order":        Filter_Order,
    "Log_Sweep":           Log_Sweep,
    "Timeout_s":           Timeout_s,
    "Output_Channel":      Output_Channel,
    "Amplitude_Range":     Amplitude_Range,
    "Output_Amplitude":    Output_Amplitude,
    "Output_Enable":       Output_Enable,
    "Output_Osc_Enable":   Output_Osc_Enable,
    "Output_Osc_Index":    Output_Osc_Index,
    "Filepath":            Filepath,
    "Filename":            Filename,
}
}")

## Run sweep with live plot

In [ ]:
%matplotlib qt
import time as _time
from collections import deque

# --- Loop expansion --------------------------------------------------------
_loop_params = {
    "Start_Freq_Hz":       Start_Freq_Hz,
    "Stop_Freq_Hz":        Stop_Freq_Hz,
    "Num_Points":          Num_Points,
    "Demod_Index":         Demod_Index,
    "Settling_Time_s":     Settling_Time_s,
    "Settling_Inaccuracy": Settling_Inaccuracy,
    "Averaging_Samples":   Averaging_Samples,
    "Bandwidth_Control":   Bandwidth_Control,
    "Bandwidth_Hz":        Bandwidth_Hz,
    "Filter_Order":        Filter_Order,
    "Log_Sweep":           Log_Sweep,
    "Timeout_s":           Timeout_s,
    "Output_Channel":      Output_Channel,
    "Amplitude_Range":     Amplitude_Range,
    "Output_Amplitude":    Output_Amplitude,
    "Output_Enable":       Output_Enable,
    "Output_Osc_Enable":   Output_Osc_Enable,
    "Output_Osc_Index":    Output_Osc_Index,
    "Filepath":            Filepath,
    "Filename":            Filename,
}

_lengths = {k: len(v) for k, v in _loop_params.items() if isinstance(v, list)}
if not _lengths:
    N = 1
else:
    _unique = set(_lengths.values())
    if len(_unique) > 1:
        raise ValueError(f"List length mismatch among loop variables: {_lengths}")
    N = _unique.pop()

def _pick(v, i):
    return v[i] if isinstance(v, list) else v

multi_file = isinstance(Filename, list)

# --- BW-normalised point counts -------------------------------------------
# Each sweep's raw points are scaled by (ref_BW / sweep_BW) so that all
# sweeps share a common "cost" unit proportional to wall-clock time.
# ref_BW cancels in the rate calculation; using the first sweep's BW keeps
# normalised counts ≈ raw counts for that sweep, making debug easier.
_ref_BW      = _pick(Bandwidth_Hz, 0)
_norm_npts   = [_pick(Num_Points, i) * _ref_BW / _pick(Bandwidth_Hz, i)
                for i in range(N)]
_total_norm  = sum(_norm_npts)
_window_norm = 0.05 * _total_norm   # rolling-window width: 5% of total norm pts

# --- Build per-iteration label for legend ----------------------------------
def _iter_label(i):
    parts = [f"Iter {i+1}"]
    amp = _pick(Output_Amplitude, i)
    parts.append(f"{amp} V")
    npts = _pick(Num_Points, i)
    parts.append(f"{int(npts)} pts")
    bw = _pick(Bandwidth_Hz, i)
    if isinstance(Bandwidth_Hz, list):
        parts.append(f"BW {bw} Hz")
    return ", ".join(parts)

# --- Figure setup ----------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(8, 6))
fig.subplots_adjust(top=0.90)
ax1.set_ylabel('R [V]')
ax2.set_ylabel('Phase [deg]')
ax2.set_xlabel('Frequency [Hz]')
_xscale = 'log' if (_pick(Log_Sweep, 0)) else 'linear'
ax1.set_xscale(_xscale)
ax2.set_xscale(_xscale)
ax1.set(yscale = "log")

_hist_colors = ['0.70', '0.55', '0.40']
_history_lines_r = deque(maxlen=3)
_history_lines_ph = deque(maxlen=3)
_history_labels = deque(maxlen=3)

_line_r, = ax1.plot([], [], 'C0-', label='current')
_line_ph, = ax2.plot([], [], 'C0-')
fig.suptitle('Ready', fontsize=10)
fig.show()

def _fmt_time(seconds):
    h, rem = divmod(int(seconds), 3600)
    m, s = divmod(rem, 60)
    if h:
        return f"{h}h {m:02d}m {s:02d}s"
    if m:
        return f"{m}m {s:02d}s"
    return f"{s}s"

# Rolling log of (wall_clock_time, cumulative_normalised_points_done).
# Entries older than _window_norm normalised points are trimmed each callback.
_rate_log = deque()
_last_elapsed   = 0.0   # elapsed time frozen until next data point arrives
_last_n_pts     = -1    # len(fv) from the previous callback; -1 forces first update

def _update_plot(partial):
    global _last_elapsed, _last_n_pts
    if partial is None:
        return
    freq = np.asarray(partial['frequency'])
    valid = freq > 0
    if not valid.any():
        return
    fv = freq[valid]
    _line_r.set_data(fv, np.asarray(partial['r'])[valid])
    _line_ph.set_data(fv, np.degrees(np.asarray(partial['phase'])[valid]))
    for ax in (ax1, ax2):
        ax.relim()
        ax.autoscale_view()

    # ── Normalised progress ─────────────────────────────────────────────────
    iter_frac = len(fv) / _current_num_points
    # Cumulative normalised points: finished iters + progress through current
    cum_norm = _completed_norm + iter_frac * _norm_npts[_current_iter - 1]
    overall_norm_frac = cum_norm / _total_norm

    now = _time.time()
    n_pts = len(fv)
    if n_pts > _last_n_pts:                      # new data point arrived
        _last_elapsed = now - _total_t_start
        _last_n_pts   = n_pts
        # ── Rolling-window rate estimate ──────────────────────────────────
        _rate_log.append((now, cum_norm))
        # Drop entries that have fallen outside the 5%-window in norm-point terms
        while len(_rate_log) > 1 and (cum_norm - _rate_log[0][1]) > _window_norm:
            _rate_log.popleft()

    title_parts = [
        f"Iter {_current_iter}/{N}",
        f"Iter: {iter_frac * 100:.0f}%",
        f"Overall: {overall_norm_frac * 100:.1f}%",
        f"Elapsed: {_fmt_time(_last_elapsed)}",
    ]

    if len(_rate_log) >= 2:
        t0, n0 = _rate_log[0]
        dt = now - t0
        dn = cum_norm - n0
        if dt > 0 and dn > 0:
            rate = dn / dt                       # normalised pts / second
            norm_remaining = _total_norm - cum_norm
            title_parts.append(f"Remaining: {_fmt_time(norm_remaining / rate)}")

    fig.suptitle("  |  ".join(title_parts), fontsize=10)
    fig.canvas.draw_idle()
    fig.canvas.flush_events()

def _push_history(result, label):
    """Move completed result into the history ring and recolour."""
    freq = np.asarray(result['frequency'])
    r_data = np.asarray(result['r'])
    ph_data = np.degrees(np.asarray(result['phase']))

    if len(_history_lines_r) == _history_lines_r.maxlen:
        _history_lines_r[0].remove()
        _history_lines_ph[0].remove()

    line_r, = ax1.plot(freq, r_data, '-', lw=1)
    line_ph, = ax2.plot(freq, ph_data, '-', lw=1)
    _history_lines_r.append(line_r)
    _history_lines_ph.append(line_ph)
    _history_labels.append(label)

    for j, (lr, lph) in enumerate(zip(_history_lines_r, _history_lines_ph)):
        c = _hist_colors[j - len(_history_lines_r)]
        lr.set_color(c)
        lph.set_color(c)
        lr.set_label(_history_labels[j])

def _save_plotly(results_so_far, labels_so_far):
    """Save an interactive Plotly HTML with all completed iterations."""
    pfig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                         vertical_spacing=0.08,
                         subplot_titles=('R [V]', 'Phase [deg]'))
    for res, lbl in zip(results_so_far, labels_so_far):
        freq = np.asarray(res['frequency'])
        pfig.add_trace(go.Scatter(x=freq, y=np.asarray(res['r']),
                                  mode='lines', name=lbl,
                                  legendgroup=lbl, showlegend=True), row=1, col=1)
        pfig.add_trace(go.Scatter(x=freq, y=np.degrees(np.asarray(res['phase'])),
                                  mode='lines', name=lbl,
                                  legendgroup=lbl, showlegend=False), row=2, col=1)
    xaxis_type = 'log' if _pick(Log_Sweep, 0) else 'linear'
    pfig.update_xaxes(type=xaxis_type)
    pfig.update_xaxes(title_text='Frequency [Hz]', row=2, col=1)
    pfig.update_yaxes(title_text='R [V]', row=1, col=1)
    pfig.update_yaxes(title_text='Phase [deg]', row=2, col=1)
    pfig.update_layout(height=700, template='plotly_white')

    if multi_file:
        html_dir = _pick(Filepath, len(results_so_far) - 1)
        html_name = os.path.splitext(_pick(Filename, len(results_so_far) - 1))[0] + '.html'
    else:
        html_dir = Filepath
        html_name = os.path.splitext(Filename)[0] + '.html'
    html_path = os.path.join(html_dir, html_name)
    os.makedirs(os.path.dirname(html_path), exist_ok=True)
    pfig.write_html(html_path)

# --- Main sweep loop -------------------------------------------------------
results = []
_iter_labels = []
_completed_norm  = 0.0          # normalised pts from fully finished iters
_total_t_start   = _time.time()
with HF2LI() as hf:
    for i in range(N):
        _current_iter      = i + 1
        _current_log_sweep = _pick(Log_Sweep, i)
        _current_num_points = _pick(Num_Points, i)
        xsc = 'log' if _current_log_sweep else 'linear'
        ax1.set_xscale(xsc)
        ax2.set_xscale(xsc)
        _line_r.set_data([], [])
        _line_ph.set_data([], [])
        _line_r.set_label(_iter_label(i))
        fig.suptitle(f'Iter {_current_iter}/{N}  |  Starting...  |  Elapsed: {_fmt_time(_time.time() - _total_t_start)}', fontsize=10)
        ax1.legend(loc='upper right', fontsize=7)
        fig.canvas.draw_idle()
        fig.canvas.flush_events()

        iter_filepath = os.path.join(_pick(Filepath, i), _pick(Filename, i))
        if multi_file:
            grp_idx = 0
            do_append = False
        else:
            grp_idx = i
            do_append = i > 0
        _last_n_pts = -1
        result = hf.sweep(
            start               = _pick(Start_Freq_Hz, i),
            stop                = _pick(Stop_Freq_Hz, i),
            num_points          = _pick(Num_Points, i),
            demod_index         = _pick(Demod_Index, i),
            settling_time       = _pick(Settling_Time_s, i),
            settling_inaccuracy = _pick(Settling_Inaccuracy, i),
            averaging_samples   = _pick(Averaging_Samples, i),
            bandwidth_control   = _pick(Bandwidth_Control, i),
            bandwidth_hz        = _pick(Bandwidth_Hz, i),
            filter_order        = _pick(Filter_Order, i),
            log_sweep           = _pick(Log_Sweep, i),
            timeout             = _pick(Timeout_s, i),
            progress            = True,
            progress_interval   = Progress_Interval_s,
            progress_callback   = _update_plot,
            output_channel      = _pick(Output_Channel, i),
            output_amplitude    = _pick(Output_Amplitude, i),
            output_enable       = _pick(Output_Enable, i),
            output_osc_enable   = _pick(Output_Osc_Enable, i),
            output_osc_index    = _pick(Output_Osc_Index, i),
            output_range        = _pick(Amplitude_Range, i),
            save                = Save,
            filepath            = iter_filepath,
            metadata            = Metadata,
            group_index         = grp_idx,
            append              = do_append,
        )
        results.append(result)
        lbl = _iter_label(i)
        _iter_labels.append(lbl)
        _push_history(result, lbl)
        ax1.legend(loc='upper right', fontsize=7)
        _save_plotly(results, _iter_labels)
        _completed_norm += _norm_npts[i]   # advance norm-point counter

total_elapsed = _time.time() - _total_t_start
fig.suptitle(f'Sweep complete  |  Total: {_fmt_time(total_elapsed)}', fontsize=10)
fig.canvas.draw_idle()
fig.canvas.flush_events()
result = results[-1]

## Final plot

In [ ]:
fig2, (fax1, fax2) = plt.subplots(2, 1, sharex=True, figsize=(8, 6))
fax1.set_ylabel('R [V]')
fax2.set_ylabel('Phase [deg]')
fax2.set_xlabel('Frequency [Hz]')
xsc = 'log' if _pick(Log_Sweep, 0) else 'linear'
fax1.set_xscale(xsc)
fax2.set_xscale(xsc)
for res, lbl in zip(results, _iter_labels):
    freq = np.asarray(res['frequency'])
    fax1.plot(freq, np.asarray(res['r']), '-', label=lbl)
    fax2.plot(freq, np.degrees(np.asarray(res['phase'])), '-')
fax1.legend(loc='upper right', fontsize=7)
fig2.suptitle('All iterations', fontsize=10)
fig2.tight_layout()
fig2.canvas.draw_idle()

## Append this sweep to the master index CSV

In [ ]:
import csv
import datetime as _dt

# Column order for the master index CSV. The header line in the file is the
# same list, joined by commas, prefixed with a single '%'.
INDEX_COLUMNS = [
    "name",
    "category",
    "frequency_span_Hz",
    "bandwidth_Hz",
    "amplitude",
    "amplitude_range",
    "amplification_dB",
    "amplification_type",
    "drive_source",
    "date",
    "notes",
    "path",
]

def append_to_index(index_file, row, columns=INDEX_COLUMNS):
    """Append `row` (dict) to `index_file`, creating it with a %-prefixed
    header if it doesn't exist yet."""
    new_file = not os.path.exists(index_file)
    os.makedirs(os.path.dirname(index_file), exist_ok=True)
    with open(index_file, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        if new_file:
            f.write("%" + ",".join(columns) + "\n")
        writer.writerow([row.get(c, "") for c in columns])

def _summarise(values, fmt=str):
    """Render a list of per-iteration values as a single CSV cell. Drops
    None entries; returns '' if empty, the single value if all equal, or
    'min-max' otherwise."""
    vals = [v for v in values if v is not None]
    if not vals:
        return ""
    uniq = sorted(set(vals))
    if len(uniq) == 1:
        return fmt(uniq[0])
    return f"{fmt(uniq[0])}-{fmt(uniq[-1])}"

def _build_row(name, rel_path, idxs):
    starts = [_pick(Start_Freq_Hz, i) for i in idxs]
    stops  = [_pick(Stop_Freq_Hz,  i) for i in idxs]
    bws    = [_pick(Bandwidth_Hz,  i) for i in idxs]
    amps   = [_pick(Output_Amplitude, i) for i in idxs]
    ranges = [_pick(Amplitude_Range, i) for i in idxs]
    freq_span = f"{min(starts)}-{max(stops)}"
    return {
        "name": name,
        "category": "Vibration Measurements",
        "frequency_span_Hz": freq_span,
        "bandwidth_Hz": _summarise(bws),
        "amplitude": _summarise(amps, fmt=lambda v: f"{v} V"),
        "amplitude_range": _summarise(ranges, fmt=lambda v: f"{v} V"),
        "amplification_dB": Amplification_dB,
        "amplification_type": Amplification_Type,
        "drive_source": Drive_Source,
        "date": _dt.date.today().isoformat(),
        "notes": Notes,
        "path": rel_path,
    }

if Append_To_Index and Save:
    if multi_file:
        # One row per output file.
        for i in range(N):
            iter_dir = _pick(Filepath, i)
            name = os.path.basename(os.path.normpath(iter_dir))
            try:
                rel_path = os.path.relpath(iter_dir, r"D:\Acoustic isolation\Data\geophone")
            except ValueError:
                rel_path = iter_dir
            row = _build_row(name, rel_path, [i])
            append_to_index(Index_File, row)
            print(f"Appended to {Index_File}: {name} ({_pick(Filename, i)})")
    else:
        # Single h5 file containing all iterations -> one row for the batch.
        name = os.path.basename(os.path.normpath(Filepath))
        try:
            rel_path = os.path.relpath(Filepath, r"D:\Acoustic isolation\Data\geophone")
        except ValueError:
            rel_path = Filepath
        row = _build_row(name, rel_path, list(range(N)))
        append_to_index(Index_File, row)
        print(f"Appended to {Index_File}: {name}")

## Read back saved HDF5 data

In [ ]:
import h5py

# with h5py.File(os.path.join(Filepath, Filename), 'r') as f:
#     grp = f[f'dev1374/demods/{Demod_Index}/sample']
#     freq  = grp['frequency'][:]
#     r     = grp['r'][:]
#     phase = grp['phase'][:]
#     x     = grp['x'][:]
#     y     = grp['y'][:]
#     print(dict(f.attrs))   # timestamp, metadata, ...

## Save notebook and copy next to data file

In [ ]:
import shutil, glob, subprocess, sys

_nb_name = "SweepZurich_V5.3.ipynb"
_nb_src  = os.path.join(os.path.dirname(os.path.abspath(_nb_name)), _nb_name)

# Save the notebook in-place via nbconvert
subprocess.run(
    [sys.executable, "-m", "jupyter", "nbconvert", "--to", "notebook",
     "--inplace", _nb_src],
    check=True,
)

# Find the actual saved data file (HF2LI prepends a timestamp to Filename)
_data_dir  = _pick(Filepath, 0) if multi_file else Filepath
_base_name = _pick(Filename, 0) if multi_file else Filename
_matches   = sorted(glob.glob(os.path.join(_data_dir, f"*{_base_name}")))

if not _matches:
    print("WARNING: no matching data file found — notebook not copied.")
else:
    _data_stem = os.path.splitext(os.path.basename(_matches[-1]))[0]
    _dest      = os.path.join(_data_dir, f"{_data_stem}_{_nb_name}")
    shutil.copy2(_nb_src, _dest)
    print(f"Copied to:\n  {_dest}")